In [1]:
import pint
import json
import jsonata
from IPython.display import JSON, HTML
import pandas as pd

# ── ANSI palette ──────────────────────────────────────────
RESET, BOLD, DIM = "\033[0m", "\033[1m", "\033[2m"
CYAN, GREEN, YELLOW = "\033[36m", "\033[32m", "\033[33m"

reg = pint.UnitRegistry()
reg.define('kg = kilogram')
reg.define('mg = milligram')
reg.define('mcg = microgram')
reg.define('mL = milliliter')
reg.define('ug = microgram')
reg.define('ng = nanogram')
reg.define('cg = centigram')
reg.define('dL = deciliter')
reg.define('L = liter')
reg.define('g = gram')
reg.define('gm = gram')
reg.define('equivalent = [substance_charge] = Eq')
reg.define('meq = 0.001 equivalent = mEq')
reg.define('mEq = 0.001 equivalent = meq')
# Activity / arbitrary units family — all dimensionally [activity]
reg.define('unit = [activity] = U = IU = iu = USP_U = arb_U')
reg.define('milliunit = 0.001 unit = mU = mIU')
reg.define('million_unit = 1e6 unit = MU = MIU')
mass = 1000 * reg.mg
vol = 1000 * reg.mL
conc = mass / vol

UNITS = ("mg/mL", "cg/dL", "ug/mL", "ng/mL", "g/mL", "mg/L", "ug/L", "g/L")


def show_conc(conc, units=UNITS, prec=6):
      """Print a concentration in multiple units as an aligned, colorized table."""
      print(f"{BOLD}{CYAN}CONC:{RESET} {BOLD}{conc:~P}{RESET}")
      w = max(map(len, units))
      for u in units:
            q = conc.to(u)
            print(f"-> {GREEN}{q.magnitude:>7,.{prec}g}{RESET}"
                  f" {RESET} {YELLOW}{u:<{w}}{RESET}")


show_conc(conc)

CONC: 1.0 mg/mL
->       1  mg/mL
->      10  cg/dL
->   1,000  ug/mL
->   1e+06  ng/mL
->   0.001  g/mL 
->   1,000  mg/L 
->   1e+06  ug/L 
->       1  g/L  


## FDA

## Functions

In [2]:
import re
import pint
from dataclasses import dataclass
from fda import clean_labeler_name

_STRENGTH_RE = re.compile(
      r'^\s*'
      r'(?P<num_val>\d+(?:\.\d+)?)\s*(?P<num_unit>[a-zA-Zµμ%]+)'
      r'\s*(?:/\s*'
      r'(?P<den_val>\d+(?:\.\d+)?)?\s*(?P<den_unit>[a-zA-Zµμ]+)'
      r')?\s*$'
)
_UCUM_BRACKET_RE = re.compile(r"\[([^\]]+)\]")
# language=JSONata
_JSONATA_PTYPE_EXTRACTOR = r"$ ~> |**[description]|{'product_type': $reverse($match(description, /\b[A-Z\s,\-]{3,}/).match)[0]}|"


def extract_product_type(obj):
      return jsonata.Jsonata(_JSONATA_PTYPE_EXTRACTOR).evaluate(obj)


def _normalize_ucum(s: str) -> str:
      # [USP'U] -> USP_U, [iU] -> iU, [arb'U] -> arb_U
      return _UCUM_BRACKET_RE.sub(
            lambda m: m.group(1).replace("'", "_").replace(".", "_"),
            s,
      )


def _normalize_denom(s: str) -> str:
      # ".9 g/100mL" -> ".9 g/(100mL)"
      return re.sub(r'/\s*(\d*\.?\d+)\s*([a-zA-Z]+)', r'/(\1\2)', s)


def parse_strength(strength_str: str) -> pint.Quantity[pint.Unit]:
      """
      Parse a pharmaceutical strength string into a pint Quantity.
      """
      if not isinstance(strength_str, str) or not strength_str.strip():
            raise ValueError(f"Empty or non-string strength: {strength_str!r}")

      s_str = strength_str.strip().replace('μ', 'u').replace('µ', 'u')  # normalize micro
      s_norm_ucum = _normalize_ucum(s_str)
      s_norm = _normalize_denom(s_norm_ucum)

      m = _STRENGTH_RE.match(s_norm)
      if not m:
            # Fall back to letting pint try directly (handles odd cases pint knows about)
            try:
                  return reg.parse_expression(s_norm)
            except Exception as e:
                  raise ValueError(f"Could not parse strength {strength_str!r}: {e}") from e

      num_val = float(m.group('num_val'))
      num_unit = m.group('num_unit')
      den_unit = m.group('den_unit')
      den_val = float(m.group('den_val')) if m.group('den_val') else 1

      try:
            numerator = num_val * reg.parse_expression(num_unit)
      except pint.errors.UndefinedUnitError as e:
            raise ValueError(f"Unknown numerator unit in {strength_str!r}: {num_unit}") from e

      if den_unit is None:
            return numerator

      try:
            denominator = den_val * reg.parse_expression(den_unit)
      except pint.errors.UndefinedUnitError as e:
            raise ValueError(f"Unknown denominator unit in {strength_str!r}: {den_unit}") from e

      if denominator.magnitude == 0:
            raise ValueError(f"Zero denominator in {strength_str!r}")

      return round(numerator / denominator, 6)


def parse_strength_in_context(obj):
      if isinstance(obj, dict):
            return {
                  k: (parse_strength(v) if k == 'strength' and isinstance(v, str)
                      else parse_strength_in_context(v))
                  for k, v in obj.items()
            }
      if isinstance(obj, list):
            return [parse_strength_in_context(x) for x in obj]
      return obj


def get_product_type(packaging):
      try:
            return packaging[0]['product_type']
      except (IndexError, KeyError, TypeError):
            return None


PACKAGE_PATTERN = re.compile(
      r"/\s*"
      r"(?P<package_size>\d+(?:\.\d+)?\s*[A-Za-zµμ]+)"
      r"\s+in\s+"
      r"(?P<package_count>\d+)\s+"
      r"(?P<package_type>[A-Za-z][A-Za-z,\- ]*?)"
      r"(?=\s*(?:\(|$))",
      re.IGNORECASE,
)


@dataclass
class PackageInfo:
      package_count: int
      package_size: str
      package_type: str


def extract_package_info(value: str) -> PackageInfo | None:
      match = PACKAGE_PATTERN.search(value)

      if not match:
            return None

      return PackageInfo(
            package_count=int(match.group("package_count")),
            package_size=" ".join(match.group("package_size").split()),
            package_type=" ".join(match.group("package_type").split()),
      )


## Usage

### Potassium Phosphates

In [4]:
from rxocrpl.fda import lookup_generic_name, lookup_ndc_package
from dataclasses import asdict
from IPython.display import JSON

kpo4 = lookup_generic_name(
      'potassium phosphates',
      dosage_form='INJECTION',
      fetch_rxcui=True,
)

JSON([asdict(product) for product in kpo4])

<IPython.core.display.JSON object>

In [ ]:
df = pd.DataFrame(kpo4)
df

In [ ]:
# language=JSONata
data = jsonata.Jsonata("""
    *.packaging[$contains(description, 'BAG')].{
        'labeler_name': %.labeler_name,
        'brand_name': %.brand_name,
        'generic_name': %.generic_name,
        'product_ndc': %.product_ndc,
        'active_ingredients': %.active_ingredients,
        'package_ndc': package_ndc,
        'description': description
    }
""").evaluate(kpo4)
df = pd.DataFrame(data)
df

In [ ]:
# language=JSONata
print('BAGS:', jsonata.Jsonata("packaging[$contains(description, 'BAG')] ~> $count()").evaluate(kpo4))
# language=JSONata
print('VIALS:', jsonata.Jsonata("packaging[$contains(description, 'VIAL')] ~> $count()").evaluate(kpo4))

In [ ]:
# language=JSONata
res = jsonata.Jsonata("""$.[
            labeler_name,
            product_ndc,
            route,
            [active_ingredients]
      ]""").evaluate(kpo4)

rows = []

for labeler_name, product_ndc, route, ingredients in res:
      for ing in ingredients:
            s = parse_strength(ing["strength"])  # pint.Quantity

            strength_mg_ml = s.to("mg/mL")

            rows.append({
                  "labeler_name": labeler_name,
                  "product_ndc": product_ndc,
                  "name": ing["name"],
                  "route": route,
                  "strength": str(s),
                  "str/mL": float(strength_mg_ml.magnitude),
                  "str_unit": str((strength_mg_ml * reg.mL).units),
            })

df = pd.DataFrame(rows)
df

### Sodium Chloride Bags

In [ ]:
ns_bag = lookup_generic_name(ingredient_names=['sodium chloride'], dosage_form='INJECTION', max_active_ingredients=1,
                             fetch_rxcui=True)
df = pd.DataFrame(ns_bag)
df = df[df["product_type"] == 'BAG']
df['strength'] = df['active_ingredients'].apply(lambda ings: parse_strength(ings[0]['strength']) if ings else None)
df['strength'] = df['strength'].apply(
      lambda s: round(s.to("mg/mL"), 6) if s is not None else None
)
df

In [ ]:
ns_bag = lookup_generic_name(ingredient_names=['sodium chloride'], dosage_form='INJECTION', max_active_ingredients=2)
df = pd.DataFrame(ns_bag)
df = df[df["product_type"] == 'BAG']
df['strength'] = df['active_ingredients'].apply(
      lambda ings: [round(parse_strength(ing['strength']), 5) for ing in ings] if ings else None)
df

### Ceftazidime-Avibactam (Avycaz)

In [ ]:
avicaz = lookup_generic_name(ingredient_names=["avibactam", "ceftazidime"], fetch_rxcui=True)
avicaz = extract_product_type(avicaz)
df = pd.DataFrame(avicaz)
df

In [ ]:
vaso = lookup_generic_name('vasopressin', dosage_form='INJECTION', fetch_rxcui=True)
data = parse_strength_in_context(vaso)
df = pd.DataFrame(data)
df

In [ ]:
unasyn = lookup_generic_name("ampicillin")
unasyn = extract_product_type(unasyn)
df = pd.DataFrame(unasyn)
df

In [ ]:
bupiv = lookup_generic_name(ingredient_names=['bupivacaine'], dosage_form="INJECTION", fetch_rxcui=True)
JSON(bupiv)

In [ ]:
result = jsonata.Jsonata("$.[active_ingredients.[name, strength]]").evaluate(vaso)

In [ ]:
dxs = jsonata.Jsonata("**.description").evaluate(vaso)
for dx in dxs:
      print(dx)
print(re.findall(r'\b[A-Z|\s|,|\-]{3,}', dx)[-1])

In [ ]:
# language=JSONata
res = jsonata.Jsonata("""$.{labeler_name:
                        [active_ingredients.{
                              'name': name,
                              'strength': strength,
                              'product_type': product_type,
                              'product_ndc': %.product_ndc
                        },
                        [**.description]]
                  }""").evaluate(vaso)
# data = parse_strength_in_context(res)
JSON(res)

### Penicillin G

In [ ]:
pen_g = lookup_generic_name("penicillin g", fetch_rxcui=True)
df = pd.DataFrame(pen_g)
df['api'] = df['active_ingredients'].apply(lambda ings: ings[0]['name'] if ings else None)
df['api_strength'] = df['active_ingredients'].apply(lambda ings: ings[0]['strength'] if ings else None)
df['api_2'] = df['active_ingredients'].apply(lambda ings: ings[1]['name'] if len(ings) > 1 else None)
df['api_2_strength'] = df['active_ingredients'].apply(lambda ings: ings[1]['strength'] if len(ings) > 1 else None)
df

In [ ]:
df[df['labeler_name'].str.contains('Roerig', na=False)]

# Filterable Drugs

In [ ]:
def get_inline_filter_drugs():
      url = r"http://pmc.ncbi.nlm.nih.gov/articles/PMC11907493/table/table1-00185787251324867"
      df = pd.read_html(url)[0]
      # remove NBSP chars
      df['Drug'] = df['Drug'].str.replace(r'\xa0', ' ')
      df['brand'] = df['Drug'].str.findall(r'\((.*?)\)')
      return df


filterable_drugs = get_inline_filter_drugs()
filterable_drugs

# RXCUI